In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os 
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Optional
from idtxl.multivariate_te import MultivariateTE
from idtxl.data import Data
from idtxl.results import Results
from idtxl.results import ResultsNetworkInference
from idtxl.stats import network_fdr
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import grangercausalitytests
import h5py
import logging
import sys
from statsmodels.stats.multitest import multipletests
import argparse

# ==============================================================================
# Configuration & Paths
# ==============================================================================
BASE_PATH = r"/root/fmri_data"
SAVE_BASE_PATH = r"/root/fmri_results"

SETTINGS = {
    'cmi_estimator': 'OpenCLKraskovCMI',
    'max_lag_sources': 3,
    'min_lag_sources': 1,
    'max_lag_target': 5,
    'alpha': 0.05,
    'n_perm_max_stat': 200,
    'n_perm_min_stat': 200,
    'n_perm_omnibus': 200,
    'n_perm_max_seq': 200,
    'verbose': True
}

def group_tasks_by_subject(all_task_files: List[str]) -> Dict[str, List[str]]:
    """Auxiliary function: Group task files by subject ID"""
    subject_task_map = {}
    for file in all_task_files:
        subject_id = file.split('_')[0]   # Extract the participant ID (e.g., '100307')
        if subject_id not in subject_task_map:
            subject_task_map[subject_id] = []
        subject_task_map[subject_id].append(file)
    return subject_task_map

def build_brain_info(file_path: str, data_df: pd.DataFrame) -> Dict[str, str]:
    """
    Extract the 'Network' information from the first row (header row) of the Excel 
    and match it with the column names (brain regions) of the data_df.
    """
    network_row = pd.read_excel(file_path, nrows=1, header=None).iloc[0]
    brain_regions = data_df.columns.tolist()
    brain_info = {}
    for col_idx, region in enumerate(brain_regions):
        # Column at index col_idx + 1 (skipping the timepoint column) matches the network
        network = network_row[col_idx + 1]
        brain_info[region] = network
    return brain_info

def apply_fdr_correction(p_matrix: np.ndarray, method: str = 'fdr_bh') -> np.ndarray:
    """Apply FDR (Benjamini-Hochberg) correction to control false positives."""
    flat_pvals = p_matrix.flatten()
    valid_mask = ~np.isnan(flat_pvals)
    valid_pvals = flat_pvals[valid_mask]   
    
    _, corrected_pvals, _, _ = multipletests(valid_pvals, alpha=0.05, method=method) 
    
    corrected_matrix = np.full_like(p_matrix, np.nan)
    corrected_matrix.flat[valid_mask] = corrected_pvals
    return corrected_matrix

def pearson_analysis(data_df: pd.DataFrame, brain_regions: List[str]) -> (np.ndarray, np.ndarray, np.ndarray):
    """Calculate the whole-brain Pearson correlation coefficient and p-value"""
    n_regions = len(brain_regions)
    corr_matrix = np.zeros((n_regions, n_regions))
    pvalue_matrix = np.zeros((n_regions, n_regions)) 
    for i, region_i in enumerate(brain_regions):
        for j, region_j in enumerate(brain_regions):
            if i <= j:
                corr, p_value = pearsonr(data_df[region_i], data_df[region_j])
                corr_matrix[i, j] = corr
                corr_matrix[j, i] = corr   # Symmetrically filled
                pvalue_matrix[i, j] = p_value   
                pvalue_matrix[j, i] = p_value   
    
    pearson_fdr_pval = apply_fdr_correction(pvalue_matrix)   
    return corr_matrix, pvalue_matrix, pearson_fdr_pval

def granger_analysis(data_df: pd.DataFrame, brain_regions: List[str], max_lag: int = 3) -> (np.ndarray, np.ndarray):
    """Calculate the whole-brain Granger causality (return the p-value matrix)"""
    n_regions = len(brain_regions)
    pvalue_matrix = np.full((n_regions, n_regions), np.nan)    
    for source_idx, source in enumerate(brain_regions):
        for target_idx, target in enumerate(brain_regions):
            if source_idx == target_idx:
                continue   # Skip self loop               
            
            ts_pair = data_df[[target, source]].values
            try:
                test_result = grangercausalitytests(ts_pair, maxlag=max_lag, verbose=False)
                p_value = test_result[max_lag][0]['ssr_ftest'][1]
                pvalue_matrix[source_idx, target_idx] = p_value   # source -> target, stored as [source, target]
            except Exception as e:
                print(f" GCA failure: {source}->{target} | {e}" )   
                
    gca_fdr_pval = apply_fdr_correction(pvalue_matrix)   
    return pvalue_matrix, gca_fdr_pval

def select_mte_sources(
    pearson_corr: np.ndarray,
    pearson_fdr_pval: np.ndarray,
    gca_fdr_pval: np.ndarray,
    brain_regions:List[str],
    pearson_threshold: float = 0.2,
    gca_threshold: float = 0.05
) -> Dict[int, List[int]]:
    """
    Pre-select candidate source regions based on Pearson and GCA results 
    to heavily optimize the IDTxl search space.
    """
    sources_dict = {}
    n_regions = len(brain_regions) 
    for i, target in enumerate(brain_regions):
        candidates = []
        for j, source in enumerate(brain_regions):
            if i == j:
                continue            
            
            pearson_positive = (abs(pearson_corr[j, i]) >= pearson_threshold and pearson_fdr_pval[j, i] <= 0.05)
            gca_positive = (not np.isnan(gca_fdr_pval[j, i]) and gca_fdr_pval[j, i] <= gca_threshold)

            if pearson_positive or gca_positive:
                candidates.append(j)     
                old_len = len(candidates)
                if old_len > 10:
                    candidates.sort(
                        key=lambda x: abs(pearson_corr[x, i]) + (-np.log10(max(gca_fdr_pval[x, i], 1e-10)) if not np.isnan(gca_fdr_pval[x, i]) else 0),
                        reverse=True
                    )
                    candidates = candidates[:10]    # Save only the top 10 strongest candidates
                    logging.info(f"Target {i} reduced candidate sources from {old_len} to 10 (keeping strongest)")
                else:
                    logging.info(f"Target {i} has {old_len} positive sources (<=10, keep all)")
        sources_dict[i] = candidates   # Target index -> List of source indices
    return sources_dict

def mte_analysis(
    data_df: pd.DataFrame,
    brain_regions: List[str],
    sources_dict: Dict[int, List[int]],
    brain_info: Dict[str, str],
    settings: Dict = SETTINGS
) -> Optional[ResultsNetworkInference]: 
  
    try:
        # [DATA FORMATTING] 'sp' dimension order (Samples, Processes) is sufficient for fMRI
        # No pseudo-trial replication is required unlike MEG
        data = Data(data_df.values, dim_order='sp')
        logging.info(f"[mTE Start analyzing {len(sources_dict)} targets]")

        mte_analysis = MultivariateTE()
        final_results = None

        for target_idx, source_idxs in sources_dict.items():
            if not source_idxs:
                continue   
            try:
                target_result = mte_analysis.analyse_single_target(settings=settings, data=data, target=target_idx, sources=source_idxs)
                if final_results is None:
                    final_results = target_result    # Initialize final_results on first pass
                else:
                    final_results.combine_results(target_result)    # Incrementally merge target data

                logging.info(f"[mTE Progress] Target {target_idx} successfully calculated.")

            except Exception as e:
                logging.warning(f"[mTE] target {target_idx} failed: {e}")   
        
        if final_results is None:
            raise ValueError("No valid mTE results after single-target analysis")

        final_results.brain_regions = brain_regions
        final_results.brain_info = brain_info

        adj = final_results.get_adjacency_matrix(weights='binary', fdr=False)
        n_links = adj.n_edges()
        logging.info(f"[mTE] Success -> {n_links} significant links (uncorrected, single_target nominal p<0.05)")

        return final_results

    except ValueError as ve:
        logging.warning(f"[mTE] No valid results: {ve}")
        return None
    except Exception as e:
        logging.error(f"[mTE] Unexpected error: {e}")
        return None

def save_results(
    save_dir: str,
    filename_prefix: str,
    result_data: Optional[Dict[str, Any]],
    brain_regions: List[str],
    brain_info: Dict[str, str],
    analysis_type: str,
    settings: Dict=SETTINGS
) -> None:
    """Save the analysis results and metadata identically structured to the MEG output."""
    if result_data is None:
        logging.warning(f"[SAVE SKIP] No result_data for {analysis_type} -> skip {filename_prefix}.h5")
        return
    
    if settings is None:
        settings = {}
    
    os.makedirs(save_dir, exist_ok=True)
    h5_path = os.path.join(save_dir, f"{filename_prefix}.h5")
    
    try:
        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('brain_regions', data=np.array(brain_regions, dtype='S'))
            network_names = [brain_info[r] for r in brain_regions]
            f.create_dataset('brain_network_map', data=np.array(network_names, dtype='S'))
            
            param_group = f.create_group('parameters')
            for k, v in settings.items():
                try:
                    param_group.attrs[k] = v
                except:
                    param_group.attrs[k] = str(v)
            param_group.attrs['analysis_type'] = analysis_type
            param_group.attrs['timestamp'] = str(pd.Timestamp.now())       
            
            result_group = f.create_group('results')
            
            if analysis_type in ["Pearson", "GCA"]:
                for key, value in result_data.items():
                    if isinstance(value, np.ndarray):
                        result_group.create_dataset(f"results/{key}", data=value)
                    elif isinstance(value, (int, float, str, bool)):
                        result_group.attrs[key] = value
                    else:
                        result_group.create_dataset(f"results/{key}", data=np.array(value, dtype='S'))
            
            elif analysis_type in ["mTE"]:
                if not isinstance(result_data, ResultsNetworkInference):
                    raise TypeError(f"Expected ResultsNetworkInference for {analysis_type}, got {type(result_data)}")
                
                mte_group = result_group.create_group("mte_results")
                
                for target in result_data.targets_analysed:
                    target_group = mte_group.create_group(f"target_{target}")
                    single_res = result_data.get_single_target(target, fdr=False)
                    
                    if "selected_vars_sources" in single_res and single_res["selected_vars_sources"]:
                        sources = np.array(single_res["selected_vars_sources"], dtype=int)
                        target_group.create_dataset("selected_sources_idx_lag", data=sources)
                    
                    if "selected_sources_te" in single_res and single_res["selected_sources_te"] is not None and len(single_res["selected_sources_te"]) > 0:
                        te_values = np.asarray(single_res["selected_sources_te"], dtype=np.float64)
                        target_group.create_dataset("te_values", data=te_values)
                    
                    if "selected_sources_pval" in single_res and single_res["selected_sources_pval"] is not None and len(single_res["selected_sources_pval"]) > 0:
                        p_values = np.asarray(single_res["selected_sources_pval"], dtype=np.float64)
                        target_group.create_dataset("p_values", data=p_values)
                    
                    for key in ["omnibus_te", "omnibus_pval", "omnibus_sign"]:
                        if key in single_res:
                            value = single_res[key]
                            if isinstance(value, (np.ndarray, np.generic)):
                                if value.ndim == 0:    # 0-dim array -> Python float/int/bool
                                    value = value.item()
                                else:
                                    value = value.tolist()    
                            elif isinstance(value, np.number):
                                value = value.item()   
                            try:
                                target_group.attrs[key] = value
                            except TypeError:
                                target_group.attrs[key] = str(value)
                
                # Adjacency matrices mapping
                adj = result_data.get_adjacency_matrix(weights='binary', fdr=False)
                binary_matrix = adj.edge_matrix.astype(np.uint8)
                result_group.create_dataset("binary_adjacency_matrix", data=binary_matrix, compression="gzip")
                
                n_nodes = len(result_data.brain_regions)
                te_weighted_matrix = np.zeros((n_nodes, n_nodes))
                for target_idx in result_data.targets_analysed:
                    single_res = result_data.get_single_target(target_idx, fdr=False)
                    if "selected_vars_sources" in single_res and single_res["selected_vars_sources"]:
                        sources = [src[0] for src in single_res["selected_vars_sources"]]
                        te_values = single_res["selected_sources_te"]
                        for src, te in zip(sources, te_values):
                            te_weighted_matrix[src, target_idx] = te
                result_group.create_dataset("te_weighted_adjacency_matrix", data=te_weighted_matrix, compression="gzip")
                
                result_group.attrs["n_significant_links"] = int(adj.n_edges())
                result_group.attrs['fdr_corrected'] = False
                result_group.attrs['note'] = 'Combined using combine_results without network-level FDR correction. te_weighted_adjacency_matrix added with significant edges weighted by TE values.'
                
                settings_group = result_group.create_group("settings")
                for k, v in result_data.settings.items():
                    try:
                        settings_group.attrs[k] = v
                    except TypeError:
                        settings_group.attrs[k] = str(v)
                
                logging.info(f"[SAVED] mTE results with {len(result_data.targets_analysed)} targets, {adj.n_edges()} significant links, TE-weighted matrix added")
            
            else:
                raise ValueError(f"Unknown analysis_type: {analysis_type}")
            
            logging.info(f"[SAVED] {h5_path}")
    except Exception as e:
        logging.error(f"[SAVE FAIL] {h5_path} | Error: {e}")
        raise

def save_mte_selection_summary(summary_dict: dict, save_path: str = None):
    if save_path is None:
        save_path = os.path.join(SAVE_BASE_PATH, "mTE_selection_summary.h5")
    
    save_dir = os.path.dirname(save_path)
    os.makedirs(save_dir, exist_ok=True)
    
    # Secure writing utilizing a temporary file mapping
    temp_path = save_path + ".tmp"
    
    with h5py.File(temp_path, 'w') as f:
        for subj_id, task_dict in summary_dict.items():
            subj_group = f.create_group(subj_id)
            for task_name, sources in task_dict.items():
                data = sources if sources else ['NONE']
                subj_group.create_dataset(task_name, data=[s.encode('utf-8') for s in data])
    
    os.replace(temp_path, save_path)  
    logging.info(f"[mTE Summary] Successfully merged and saved: {save_path} ({len(summary_dict)} subjects)")

def parse_subject_selection(subj_input: str, all_subj_ids: List[str]) -> List[str]:
    """Parse subject selection (single ID, range, or multiple IDs)."""
    selected = []
    if not subj_input or subj_input.lower() == 'all':
        return all_subj_ids
    
    for part in subj_input.split(','):
        part = part.strip()
        if '-' in part and part.replace('-', '').isdigit():
            try:
                start, end = map(int, part.split('-'))
                start = max(0, start)
                end = min(len(all_subj_ids)-1, end)
                selected.extend(all_subj_ids[i] for i in range(start, end+1))
            except:
                logging.warning(f"Invalid range: {part}, skipping")
        elif part in all_subj_ids:
            selected.append(part)
        else:
            logging.warning(f"Subject ID {part} not found, skipping")
    
    return list(set(selected))  

def run_one_task(task_tuple):
    subj_id, task_name, task_file = task_tuple
    file_path = os.path.join(BASE_PATH, task_file)
    
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format=f'%(asctime)s | %(levelname)-8s | PID{{os.getpid():5d}} | SUBJ {subj_id} | TASK {task_name} | %(message)s',
        handlers=[
            logging.FileHandler(f"{SAVE_BASE_PATH}/log_{subj_id}_{task_name}.log", encoding='utf-8'),
            logging.StreamHandler(sys.stdout)
        ],
        force=True
    )
    
    logging.info(f"START → Subject {subj_id} | Task {task_name}")
    
    local_summary = {} 
    
    
    # header=[0,1] sets Network & Region as multi-index column headers
    # skiprows=[2] bypasses the 0-indexed integer header created by Pandas
    data_df = pd.read_excel(file_path, header=[0,1], skiprows=[2], index_col=0)
    data_df.dropna(axis=1, how='all', inplace=True)    # Secure padding deletion for empty columns
    data_df.columns = [f"{net}_{reg}" for net, reg in data_df.columns]
    brain_info = build_brain_info(file_path, data_df)
    brain_regions = list(data_df.columns)
    
    # 1. Pearson Analysis
    corr_matrix, pearson_pval, pearson_fdr_pval = pearson_analysis(data_df, brain_regions)
    save_results(SAVE_BASE_PATH, f"Pearson_{subj_id}_{task_name}", 
                 {'corr_matrix': corr_matrix, 'pval_matrix': pearson_pval, 'pearson_fdr_pval': pearson_fdr_pval},
                 brain_regions, brain_info, 'Pearson')
    
    # 2. Granger Causality Analysis (GCA)
    gca_pval, gca_fdr_pval = granger_analysis(data_df, brain_regions)
    save_results(SAVE_BASE_PATH, f"GCA_{subj_id}_{task_name}",
                 {'pval_matrix': gca_pval, 'gca_fdr_pval': gca_fdr_pval},
                 brain_regions, brain_info, 'GCA')
    
    # 3. mTE Source Selection (Optimization filter)
    sources_dict = select_mte_sources(corr_matrix, pearson_fdr_pval, gca_fdr_pval, brain_regions)
    logging.info(f"→ mTE will analyze {len(sources_dict)} targets")
    local_summary[task_name] = list(sources_dict.keys()) if sources_dict else []
    
    # 4. mTE Analysis
    if sources_dict:
        mte_results = mte_analysis(data_df, brain_regions, sources_dict, brain_info)
        save_results(SAVE_BASE_PATH, f"mTE_{subj_id}_{task_name}", mte_results,
                     brain_regions, brain_info, 'mTE')
    else:
        logging.info("→ No candidate sources, skipping mTE")
    
    logging.info(f"FINISH → Subject {subj_id} | Task {task_name}")
    
    return local_summary


if __name__ == "__main__":
    import argparse
    from multiprocessing import Pool
    
    parser = argparse.ArgumentParser(description="HCP Brain Fingerprint - Task-Level Parallel")
    parser.add_argument('--subj', type=str, required=True,
                        help="Subject selection: 'all', single ID (100307), or comma-separated")
    parser.add_argument('--parallel', type=int, default=1,
                        help="Max parallel tasks (recommend 40-80 for your server)")
    args = parser.parse_args()
    
    os.makedirs(SAVE_BASE_PATH, exist_ok=True)
    
    all_task_files = sorted([f for f in os.listdir(BASE_PATH) if f.endswith('.xlsx')])
    subject_task_map = group_tasks_by_subject(all_task_files)
    all_subj_ids = sorted(subject_task_map.keys())
    selected_subj_ids = parse_subject_selection(args.subj, all_subj_ids)
    
    tasks = []
    for subj_id in selected_subj_ids:
        for task_file in subject_task_map[subj_id]:
            task_name = task_file.split('.')[0]
            tasks.append((subj_id, task_name, task_file))
    
    print(f"Total {len(tasks)} tasks to run with up to {args.parallel} parallel workers")
    
    all_summary = {}  
    
    if args.parallel > 1 and len(tasks) > 1:
        print(f"Launching {args.parallel} parallel workers for {len(tasks)} tasks")
        with Pool(args.parallel) as p:
            for task_tuple, task_summary in zip(tasks, p.imap_unordered(run_one_task, tasks)):
                subj_id = task_tuple[0]
                task_name = task_tuple[1]
                if subj_id not in all_summary:
                    all_summary[subj_id] = {}
                all_summary[subj_id][task_name] = task_summary
    else:
        for t in tasks:
            subj_id = t[0]
            task_name = t[1]
            task_summary = run_one_task(t)
            if subj_id not in all_summary:
                all_summary[subj_id] = {}
            all_summary[subj_id][task_name] = task_summary

    if all_summary:
        save_mte_selection_summary(all_summary)
        print(f"mTE selection summary saved to mTE_selection_summary.h5 ({len(all_summary)} subjects)")
    else:
        print("No mTE selection data to save.")

    print("ALL TASKS COMPLETED!")